# IE4422: ScienceQA → Qwen → progressive hints

Run this notebook in **Google Colab**, using a **GPU runtime**. Your own computer needs no GPU.
Keep `IE4422_Source.zip` beside your downloaded notebook; upload that ZIP when prompted.
Choose **Runtime → Change runtime type → GPU**, then run the cells in order.

Defaults: save to your own Drive, prepare/restore ScienceQA, download/restore a fixed Qwen checkpoint,
and generate three hints for one validation question. You will enter the student's response after seeing the question.
The first run downloads about 626 MB of data and 7.5 GB of model weights. Allow 8 GB or more free Drive space
for model caching, plus data/results. No Hugging Face account is required for these public files.

This is the existing exploratory **evidence → student-response diagnosis → hints** prototype.
Outputs may be wrong or disclose answers. It does not implement the final response-independent A–E research protocol.
Reference answers and explanations stay outside generation. Use train/validation only for this demo.

The notebook launches all project jobs in **one project environment**, inheriting Colab's working CUDA wheels.
It does not replace the notebook kernel's packages or require a kernel restart.


## 1. Check the cloud runtime

A local GPU or Python installation is unnecessary. If CUDA is unavailable, select a GPU runtime and reconnect.
Colab does not guarantee a particular GPU or session length.


In [ ]:
import json
import os
from pathlib import Path, PurePosixPath
import shutil
import stat
import subprocess
import sys
import zipfile

if not Path('/content').is_dir() or not sys.platform.startswith('linux'):
    raise RuntimeError('Open this notebook in Google Colab, not a local notebook kernel.')
if sys.version_info < (3, 12):
    raise RuntimeError('Select a current Colab Python 3.12+ runtime for the shared dependency pins.')
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime → Change runtime type → GPU, then reconnect and rerun.')
print('Python:', sys.version.split()[0], '| PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
print('Local disk free (GiB):', round(shutil.disk_usage('/content').free / 2**30, 1))


## 2. Upload and extract the source ZIP

Upload the updated `IE4422_Source.zip`. Leave `UPLOAD_NEW_SOURCE=True` when replacing an older bundle;
set it to False only when deliberately reusing the uploaded ZIP in the same runtime.
Source runs from `/content/IE4422`; all paths below are set automatically.


In [ ]:
WORK = Path('/content/IE4422')
SOURCE_ZIP = Path('/content/IE4422_Source.zip')
UPLOAD_NEW_SOURCE = True  # @param {type:"boolean"}
if UPLOAD_NEW_SOURCE or not SOURCE_ZIP.is_file():
    from google.colab import files
    uploaded = files.upload()
    archives = [name for name in uploaded if name.lower().endswith('.zip')]
    if len(archives) != 1:
        raise ValueError('Upload exactly one IE4422 source ZIP.')
    SOURCE_ZIP.write_bytes(uploaded[archives[0]])

# Validate paths before extracting the package. The ZIP contains source only.
with zipfile.ZipFile(SOURCE_ZIP) as source:
    names = set()
    for info in source.infolist():
        path = PurePosixPath(info.filename)
        if (not path.parts or path.parts[0] != 'IE4422' or path.is_absolute()
                or '..' in path.parts or '\\' in info.filename or ':' in info.filename
                or info.filename in names or stat.S_ISLNK(info.external_attr >> 16)
                or not (Path('/content') / info.filename).resolve().is_relative_to(WORK.resolve())):
            raise ValueError(f'Unexpected ZIP member: {info.filename}')
        names.add(info.filename)
    if 'IE4422/bundle_info.json' not in names:
        raise ValueError('Use the supplied IE4422_Source.zip, not a general repository download.')
    source.extractall('/content')
sys.path.insert(0, str(WORK))
import importlib
import notebooks.colab_support as storage
importlib.reload(storage)  # Pick up the helper from the newly uploaded bundle.
from notebooks.colab_support import (read_json, write_json, archive_prepared,
                                    restore_prepared, prepared_signature, run_visible)
bundle = read_json(WORK / 'bundle_info.json')
print('Source extracted:', WORK)
print('Bundle created:', bundle['created_at_utc'])


## 3. Choose persistence settings

Leave these defaults for a reusable setup. Approve mounting **your own** Drive.
Work is performed locally; Drive retains a prepared-data archive, optional checkpoint, and results.
`REBUILD_DATA=True` repeats preparation. With Drive disabled, temporary data/model files are lost when the runtime ends.


In [ ]:
USE_DRIVE = True  # @param {type:"boolean"}
SAVE_MODEL_TO_DRIVE = True  # @param {type:"boolean"}
REBUILD_DATA = False  # @param {type:"boolean"}
PERSISTENT = None
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PERSISTENT = Path('/content/drive/MyDrive/IE4422')
    PERSISTENT.mkdir(parents=True, exist_ok=True)
print('Persistent folder:', PERSISTENT or 'disabled; download results at the end')


## 4. Set up one project environment

This may take several minutes on the first run. The helper preserves Colab's installed CUDA torch/torchvision
and respects their active dependency limits, applies compatible shared project pins, checks the project's dependency graph,
and verifies CUDA inside the project environment. Environment creation skips `ensurepip`; Colab's existing pip explicitly targets the project Python.
Each command's output and errors are displayed live. It does not alter the notebook kernel or silently switch to CPU.
Local Windows setup remains unchanged. Actual Colab package versions are saved with each generated result.


In [ ]:
ENVIRONMENT = Path('/content/ie4422_env')
run_visible([sys.executable, str(WORK / 'notebooks/colab_support.py'),
                'setup', str(WORK), str(ENVIRONMENT)], cwd=WORK)
PYTHON = ENVIRONMENT / 'bin/python'
print('Project Python:', PYTHON)


## 5. Check the packaged code

These focused tests use synthetic inputs/fake tutor calls. They check data handling, reference separation,
model reuse, caching and packaging; passing them does not establish model quality or real GPU feasibility.


In [ ]:
run_visible([str(PYTHON), '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=WORK)


## 6. Download/prepare ScienceQA, or restore a matching prepared copy

The existing `prepare_data.py` performs the download, filtering and validation.
Its local cache setting is left intact. The Drive archive stores exported questions, separate references,
images, manifests and audits; it does not store the raw Hugging Face download cache.
Source/configuration mismatches stop with an instruction to rebuild rather than silently reuse stale data.


In [ ]:
signature = prepared_signature(WORK)
local_marker = WORK / 'tmp/prepared_signature.json'
local_archive = WORK / 'tmp/scienceqa_prepared.zip'
exports = [WORK / f'data/processed/scienceqa/{folder}/{split}.jsonl'
           for folder in ('questions', 'references') for split in ('train', 'validation', 'test')]
ready = (not REBUILD_DATA and local_marker.is_file()
         and read_json(local_marker) == signature and all(path.is_file() for path in exports))
if not ready and not REBUILD_DATA and PERSISTENT and (PERSISTENT / local_archive.name).is_file():
    local_archive.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(PERSISTENT / local_archive.name, local_archive)
    ready = restore_prepared(WORK, local_archive)
    print('Restored prepared data from Drive.')
if not ready:
    run_visible([str(PYTHON), 'prepare_data.py'], cwd=WORK)
    archive_prepared(WORK, local_archive)
    if PERSISTENT:
        pending = PERSISTENT / (local_archive.name + '.partial')
        shutil.copy2(local_archive, pending)
        pending.replace(PERSISTENT / local_archive.name)
    print('Prepared data and saved a reusable archive.')
else:
    print('Using matching prepared exports.')
write_json(local_marker, signature)
audit = read_json(WORK / 'reports/scienceqa/integrity_audit.json')
print('Valid counts:', audit['valid_counts'])


## 7. Inspect one development question

Leave `ITEM_ID` empty to use the first valid item from the chosen development split.
This cell displays only the public question, supplied context, choices and original image.
The metadata filter only establishes image presence; choose a reviewed, diagram-dependent development item for a feasibility trial.


In [ ]:
SPLIT = 'validation'  # @param ["train", "validation"]
ITEM_ID = ''  # @param {type:"string"}
if SPLIT not in ('train', 'validation'):
    raise ValueError('This development notebook does not run on held-out test items.')
from run_scienceqa_tutor import load_question
if not ITEM_ID.strip():
    ITEM_ID = read_json(WORK / f'manifests/scienceqa/{SPLIT}_valid_ids.json')[0]
item = load_question(WORK, SPLIT, ITEM_ID.strip())
print(item['item_id'], '\n', item['question'])
if item['context'].strip():
    print('Supplied context:', item['context'])
for choice in item['choices']:
    print(f"{choice['id']}. {choice['text']}")
from IPython.display import Image, display
display(Image(filename=item['image_path']))


## 8. Enter the student's response

Enter an actual attempt for the displayed question, not a reference answer copied from an export.
The prototype still uses response diagnosis; its behaviour has not been validated as a tutor.


In [ ]:
STUDENT_ANSWER = input('Student answer (option letter or text): ').strip()
STUDENT_EXPLANATION = input('Student reasoning: ').strip()
HINT_LEVELS = [1, 2, 3]
if not STUDENT_ANSWER or not STUDENT_EXPLANATION:
    raise ValueError('Both a student response and reasoning are required.')
ATTEMPT_FILE = WORK / 'tmp/student_attempt.json'
write_json(ATTEMPT_FILE, {'answer': STUDENT_ANSWER, 'explanation': STUDENT_EXPLANATION})


## 9. Download/restore the fixed Qwen checkpoint

Drive **can** hold model weights. A complete materialized checkpoint is copied to local storage before loading.
No model weights are included in the source ZIP. Downloading original weights and loading in 4-bit are separate steps.
Rerun this cell after an interrupted transfer; incomplete checkpoints are not treated as ready.


In [ ]:
model_command = [str(PYTHON), str(WORK / 'notebooks/colab_support.py'), 'model', str(WORK)]
if PERSISTENT and SAVE_MODEL_TO_DRIVE:
    model_command.extend(['--persistent', str(PERSISTENT / 'models')])
run_visible(model_command, cwd=WORK)


## 10. Generate and save the hints

One model instance serves evidence extraction, diagnosis and every requested hint in this invocation.
The demo reuses its raw evidence/diagnosis strings for this one item (five model calls for three hints).
Each completed stage is saved locally and, when enabled, to Drive. Failures remain visible and partial files remain available.

This is an offline prototype demonstration: preceding hints are not included in later prompts, no disclosure checker
is applied, and the strings are not schema-validated. It is not an A–E evaluation run.
The worker exits after generation and releases its GPU memory; rerunning this cell starts a new model instance.


In [ ]:
RUNS = WORK / 'outputs/colab'
previous_runs = set(RUNS.glob('*')) if RUNS.exists() else set()
command = [str(PYTHON), str(WORK / 'notebooks/run_demo.py'), '--split', SPLIT,
           '--item-id', item['item_id'], '--attempt', str(ATTEMPT_FILE),
           '--levels', *map(str, HINT_LEVELS), '--output-dir', str(RUNS)]
if PERSISTENT:
    command.extend(['--persistent', str(PERSISTENT / 'runs')])
run_visible(command, cwd=WORK)
new_runs = set(RUNS.glob('*')) - previous_runs
if len(new_runs) != 1:
    raise RuntimeError('Expected one new run directory; inspect outputs/colab.')
LATEST_RUN = new_runs.pop()
result = read_json(LATEST_RUN / 'result.json')
run_record = read_json(LATEST_RUN / 'run.json')
if run_record['status'] != 'complete':
    raise RuntimeError('This run did not complete; inspect the saved partial outputs.')
for level, hint in result['hints'].items():
    print(f'\nHint {level}: {hint}')
print('\nLocal results:', LATEST_RUN)
if PERSISTENT:
    print('Drive results:', PERSISTENT / 'runs' / LATEST_RUN.name)
print('Generation seconds:', round(run_record['generation_seconds'], 2))


## 11. Download this run (optional)

Drive users already have a persistent copy. Run this cell if you also want a small ZIP of this run on your computer.
It contains outputs and settings, not the model or dataset. Previous runs are kept in separate folders.


In [ ]:
from google.colab import files
export = shutil.make_archive(str(WORK / 'tmp' / LATEST_RUN.name), 'zip', root_dir=LATEST_RUN)
files.download(export)


For a different item, rerun cells 7–10. To repeat dataset preparation, set `REBUILD_DATA=True` and rerun cell 6.
Model weights and matching prepared data can be restored after reconnecting, but the environment must be set up again.
Keep outputs for failures and avoid using test examples for prompt development.

Sources: [Colab storage/resource behaviour](https://research.google.com/colaboratory/faq.html),
[Hugging Face checkpoint downloads](https://huggingface.co/docs/huggingface_hub/guides/download).
Actual CUDA inference in Colab still requires verification on the assigned hardware.
